In [ ]:
!pip install -q cma

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    roc_curve, auc, classification_report
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

import cma

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/kaggle/input/liver-histopathology"  # flat class-folder dataset: F0, F1, F2, F3, F4
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 5  # F0, F1, F2, F3, F4 (liver fibrosis stages)
CLASS_NAMES = None  # filled after loading dataset


In [ ]:

from sklearn.model_selection import train_test_split

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Dataset is flat class folders (F0..F4) -> no pre-made Training/Testing split,
# so we load once with ImageFolder and split 70/15/15 ourselves (stratified by class via indices).
full_dataset = datasets.ImageFolder(DATA_DIR, transform=eval_transform)
CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

targets = np.array(full_dataset.targets)
train_idx, temp_idx = train_test_split(
    np.arange(len(targets)), test_size=0.30, stratify=targets, random_state=SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=targets[temp_idx], random_state=SEED
)

train_dataset = torch.utils.data.Subset(
    datasets.ImageFolder(DATA_DIR, transform=train_transform), train_idx
)
val_dataset = torch.utils.data.Subset(full_dataset, val_idx)
test_dataset = torch.utils.data.Subset(full_dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Classes: {CLASS_NAMES}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")


# --- Class weights for imbalance handling (F0..F4 liver fibrosis stages) ---
def compute_class_weights(target_array, num_classes):
    counts = np.bincount(target_array, minlength=num_classes)
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)

CLASS_WEIGHTS = compute_class_weights(targets[train_idx], NUM_CLASSES).to(DEVICE)
print("Class distribution (train):", np.bincount(targets[train_idx], minlength=NUM_CLASSES))
print("Class weights (inverse frequency):", CLASS_WEIGHTS.cpu().numpy())


In [ ]:
ORDINAL_LAMBDA = 0.3   # weight of the ordinal-distance penalty
FOCAL_GAMMA = 2.0      # focusing parameter for focal loss


class FocalLoss(nn.Module):
    """Cross-entropy with class weights (imbalance) + a focusing term (hard examples)."""
    def __init__(self, class_weights=None, gamma=2.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


def ordinal_distance_penalty(logits, targets, num_classes):
    """F0..F4 are ordered stages, not independent categories. This penalizes
    predictions that are ordinally far from the true stage (e.g. predicting F4
    when the truth is F0 is worse than predicting F1), on top of plain CE which
    treats every wrong class equally."""
    probs = F.softmax(logits, dim=1)
    class_indices = torch.arange(num_classes, device=logits.device).float()
    expected_idx = (probs * class_indices).sum(dim=1)
    true_idx = targets.float()
    return F.mse_loss(expected_idx, true_idx)


focal_loss_fn = FocalLoss(class_weights=CLASS_WEIGHTS, gamma=FOCAL_GAMMA)


def domain_loss(logits, targets, num_classes):
    return focal_loss_fn(logits, targets) + ORDINAL_LAMBDA * ordinal_distance_penalty(logits, targets, num_classes)


In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        y = torch.cat([avg_out, max_out], dim=1)
        y = self.conv(y)
        return self.sigmoid(y)


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x

In [ ]:
def build_teacher(name, num_classes):
    if name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    else:
        raise ValueError(f"Unknown teacher {name}")
    return model


def train_one_teacher(model, train_loader, val_loader, epochs=5, lr=1e-4):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                preds = out.argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        val_acc = correct / total
        print(f"[Teacher {model.__class__.__name__}] Epoch {epoch+1}/{epochs} "
              f"train_loss={train_loss:.4f} val_acc={val_acc:.4f}")
    return model


class TeacherEnsemble(nn.Module):
    """Wraps multiple trained teachers, returns averaged softmax probabilities."""
    def __init__(self, teachers, weights=None):
        super().__init__()
        self.teachers = nn.ModuleList(teachers)
        for t in self.teachers:
            t.eval()
            for p in t.parameters():
                p.requires_grad = False
        n = len(teachers)
        self.weights = weights if weights is not None else [1.0 / n] * n

    @torch.no_grad()
    def forward(self, x):
        probs = 0
        for w, t in zip(self.weights, self.teachers):
            logits = t(x)
            probs = probs + w * F.softmax(logits, dim=1)
        return probs  # already probabilities, summed to ~1

In [ ]:
class StudentModel(nn.Module):
    def __init__(self, num_classes, use_cbam=True):
        super().__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        out_channels = 1280  # efficientnet_b0 final feature channels

        self.use_cbam = use_cbam
        if use_cbam:
            self.attention = CBAM(out_channels)
        else:
            self.attention = SEBlock(out_channels)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(out_channels, num_classes)

    def forward(self, x, return_features=False):
        x = self.features(x)
        x = self.attention(x)
        x = self.pool(x)
        feat = torch.flatten(x, 1)
        out = self.classifier(self.dropout(feat))
        if return_features:
            return out, feat
        return out

In [ ]:
def distillation_loss(student_logits, teacher_probs, true_labels, alpha, temperature):
    ce_loss = domain_loss(student_logits, true_labels, NUM_CLASSES)  # focal + ordinal, replaces plain CE

    student_log_soft = F.log_softmax(student_logits / temperature, dim=1)
    teacher_soft = teacher_probs  # already softmax probabilities from ensemble
    # sharpen/soften teacher probs with temperature by re-deriving via log then softmax
    teacher_soft_T = F.softmax(torch.log(teacher_soft + 1e-8) / temperature, dim=1)

    kd_loss = F.kl_div(student_log_soft, teacher_soft_T, reduction="batchmean") * (temperature ** 2)

    return alpha * kd_loss + (1 - alpha) * ce_loss

In [ ]:
def train_student(student, teacher_ensemble, train_loader, val_loader,
                   alpha, temperature, lr, epochs, verbose=True):
    student = student.to(DEVICE)
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        student.train()
        running_loss, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            student_logits = student(x)
            with torch.no_grad():
                teacher_probs = teacher_ensemble(x)
            loss = distillation_loss(student_logits, teacher_probs, y, alpha, temperature)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)
            preds = student_logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = correct / total

        student.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                student_logits = student(x)
                teacher_probs = teacher_ensemble(x)
                loss = distillation_loss(student_logits, teacher_probs, y, alpha, temperature)
                val_running_loss += loss.item() * x.size(0)
                preds = student_logits.argmax(dim=1)
                val_correct += (preds == y).sum().item()
                val_total += y.size(0)

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if verbose:
            print(f"Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} "
                  f"train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    return student, history

In [ ]:
def cma_es_objective(params, teacher_ensemble, train_loader, val_loader):
    """params = [alpha_raw, temperature_raw, lr_raw] in unconstrained space,
    mapped into valid ranges."""
    alpha = 1 / (1 + np.exp(-params[0]))            # sigmoid -> (0,1)
    temperature = 1.0 + 9.0 * (1 / (1 + np.exp(-params[1])))  # -> (1,10)
    lr = 10 ** (-5 + 3 * (1 / (1 + np.exp(-params[2]))))      # -> (1e-5, 1e-2)

    student = StudentModel(NUM_CLASSES, use_cbam=True)
    _, history = train_student(
        student, teacher_ensemble, train_loader, val_loader,
        alpha=alpha, temperature=temperature, lr=lr, epochs=2, verbose=False
    )
    val_loss = history["val_loss"][-1]
    print(f"  tried alpha={alpha:.3f} temp={temperature:.3f} lr={lr:.2e} -> val_loss={val_loss:.4f}")
    return val_loss


def run_cma_es_search(teacher_ensemble, train_loader, val_loader, max_iters=8, popsize=4):
    x0 = [0.0, 0.0, 0.0]
    sigma0 = 0.8
    es = cma.CMAEvolutionStrategy(x0, sigma0, {"popsize": popsize, "maxiter": max_iters, "verbose": -9})

    while not es.stop():
        solutions = es.ask()
        fitnesses = [
            cma_es_objective(sol, teacher_ensemble, train_loader, val_loader)
            for sol in solutions
        ]
        es.tell(solutions, fitnesses)
        es.disp()

    best = es.result.xbest
    best_alpha = 1 / (1 + np.exp(-best[0]))
    best_temp = 1.0 + 9.0 * (1 / (1 + np.exp(-best[1])))
    best_lr = 10 ** (-5 + 3 * (1 / (1 + np.exp(-best[2]))))
    print(f"Best found -> alpha={best_alpha:.4f}, temperature={best_temp:.4f}, lr={best_lr:.6f}")
    return best_alpha, best_temp, best_lr

In [ ]:
teacher1 = build_teacher("resnet50", NUM_CLASSES)
teacher1 = train_one_teacher(teacher1, train_loader, val_loader, epochs=5, lr=1e-4)

teacher2 = build_teacher("densenet121", NUM_CLASSES)
teacher2 = train_one_teacher(teacher2, train_loader, val_loader, epochs=5, lr=1e-4)

teacher_ensemble = TeacherEnsemble([teacher1, teacher2]).to(DEVICE)

In [ ]:
best_alpha, best_temp, best_lr = run_cma_es_search(
    teacher_ensemble, train_loader, val_loader, max_iters=6, popsize=4
)

final_student = StudentModel(NUM_CLASSES, use_cbam=True)
final_student, history = train_student(
    final_student, teacher_ensemble, train_loader, val_loader,
    alpha=best_alpha, temperature=best_temp, lr=best_lr, epochs=20, verbose=True
)

torch.save(final_student.state_dict(), "/kaggle/working/student_model.pth")

In [ ]:
final_student.eval()
all_preds, all_labels, all_probs, all_feats = [], [], [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        logits, feats = final_student(x, return_features=True)
        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_feats.extend(feats.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
all_feats = np.array(all_feats)

In [ ]:
acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, support = precision_recall_fscore_support(
    all_labels, all_preds, average=None, labels=range(NUM_CLASSES)
)
precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
    all_labels, all_preds, average="weighted"
)

print(f"Overall Accuracy: {acc:.4f}")
print(f"Weighted Precision: {precision_w:.4f} | Weighted Recall: {recall_w:.4f} | Weighted F1: {f1_w:.4f}\n")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

metrics_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    "Support": support
})
print(metrics_df)

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150)
plt.show()

FP = cm.sum(axis=0) - np.diag(cm)
FN = cm.sum(axis=1) - np.diag(cm)
TP = np.diag(cm)
TN = cm.sum() - (FP + FN + TP)

TPR = TP / (TP + FN)        # Sensitivity / Recall
TNR = TN / (TN + FP)        # Specificity
FPR = FP / (FP + TN)
FNR = FN / (FN + TP)

rates_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "TPR (Sensitivity)": TPR,
    "TNR (Specificity)": TNR,
    "FPR": FPR,
    "FNR": FNR
})
print(rates_df)

In [ ]:
labels_bin = label_binarize(all_labels, classes=range(NUM_CLASSES))

plt.figure(figsize=(7, 6))
for i in range(NUM_CLASSES):
    fpr_i, tpr_i, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
    roc_auc_i = auc(fpr_i, tpr_i)
    plt.plot(fpr_i, tpr_i, label=f"{CLASS_NAMES[i]} (AUC = {roc_auc_i:.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (One-vs-Rest)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("/kaggle/working/roc_curves.png", dpi=150)
plt.show()

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label="Train Loss")
plt.plot(epochs_range, history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_acc"], label="Train Accuracy")
plt.plot(epochs_range, history["val_acc"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()

plt.tight_layout()
plt.savefig("/kaggle/working/loss_accuracy_curves.png", dpi=150)
plt.show()

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca")
feats_2d = tsne.fit_transform(all_feats)

plt.figure(figsize=(7, 6))
for i in range(NUM_CLASSES):
    idx = all_labels == i
    plt.scatter(feats_2d[idx, 0], feats_2d[idx, 1], label=CLASS_NAMES[i], s=10, alpha=0.7)
plt.legend()
plt.title("t-SNE of Student Model Features (Test Set)")
plt.tight_layout()
plt.savefig("/kaggle/working/tsne.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# Performance Comparison: Under-Sampling vs Over-Sampling
# (vs your current Class-Weighted approach as reference)
# Trains the proposed student model (KD + CMA-ES tuned) with
# each resampling strategy, keeping everything else fixed.
# ============================================================

!pip install -q imbalanced-learn
from collections import Counter

RESAMPLE_EPOCHS = 20   # match your final run for reportable numbers

# ---- get all (path/index, label) info from the training set ----
if hasattr(train_loader.dataset, "indices"):  # torch.utils.data.Subset
    base_dataset = train_loader.dataset.dataset
    base_indices = list(train_loader.dataset.indices)
else:
    base_dataset = train_loader.dataset
    base_indices = list(range(len(base_dataset)))

all_train_targets = np.array([base_dataset.targets[i] for i in base_indices])
print("Original class distribution:", Counter(all_train_targets))


def build_resampled_indices(strategy):
    """Returns a list of dataset indices resampled according to strategy:
    'under' = random under-sampling to the minority class count
    'over'  = random over-sampling (with replacement) to the majority class count
    'none'  = original, unmodified (class-weighted baseline for reference)"""
    if strategy == "none":
        return base_indices

    class_to_indices = {c: [base_indices[i] for i in range(len(base_indices)) if all_train_targets[i] == c]
                         for c in range(NUM_CLASSES)}
    counts = {c: len(idxs) for c, idxs in class_to_indices.items()}

    if strategy == "under":
        target_n = min(counts.values())
        rng = np.random.RandomState(SEED)
        resampled = []
        for c, idxs in class_to_indices.items():
            resampled.extend(rng.choice(idxs, size=target_n, replace=False))
    elif strategy == "over":
        target_n = max(counts.values())
        rng = np.random.RandomState(SEED)
        resampled = []
        for c, idxs in class_to_indices.items():
            resampled.extend(rng.choice(idxs, size=target_n, replace=True))
    else:
        raise ValueError(f"Unknown strategy {strategy}")

    rng = np.random.RandomState(SEED)
    rng.shuffle(resampled)
    return resampled


def train_with_resampling(strategy, use_class_weights=False):
    """strategy: 'under', 'over', or 'none' (class-weighted, no resampling)."""
    resampled_indices = build_resampled_indices(strategy)
    resampled_targets = np.array([base_dataset.targets[i] for i in resampled_indices])
    print(f"\n=== Strategy: {strategy} -> {len(resampled_indices)} samples, "
          f"distribution: {Counter(resampled_targets)} ===")

    resampled_ds = torch.utils.data.Subset(base_dataset, resampled_indices)
    resampled_loader = DataLoader(resampled_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    student_r = StudentModel(NUM_CLASSES, use_cbam=True)

    if use_class_weights:
        # reuse your normal pipeline (train_student already applies CLASS_WEIGHTS internally)
        student_r, _ = train_student(
            student_r, teacher_ensemble, resampled_loader, val_loader,
            alpha=best_alpha, temperature=best_temp, lr=best_lr,
            epochs=RESAMPLE_EPOCHS, verbose=False
        )
    else:
        # for under/over-sampling, classes are already balanced -> use UNWEIGHTED loss
        # via the ablation loss builder (use_focal=True keeps focal loss but with
        # uniform class weights since resampling already balanced the data)
        loss_fn = make_ablation_loss_fn(
            use_kd=True, use_focal=True, use_ordinal=True,
            class_weights=None, focal_gamma=FOCAL_GAMMA, ordinal_lambda=ORDINAL_LAMBDA
        )
        student_r = train_student_ablation(
            student_r, teacher_ensemble, resampled_loader, val_loader, loss_fn,
            use_kd=True, alpha=best_alpha, temperature=best_temp, lr=best_lr,
            epochs=RESAMPLE_EPOCHS
        )

    return student_r


# ---- run all 3 strategies ----
resample_configs = [
    {"name": "Class-Weighted (current/reference)", "strategy": "none", "use_class_weights": True},
    {"name": "Under-Sampling", "strategy": "under", "use_class_weights": False},
    {"name": "Over-Sampling", "strategy": "over", "use_class_weights": False},
]

resample_results = []
resample_cms = {}

for cfg in resample_configs:
    model_r = train_with_resampling(cfg["strategy"], cfg["use_class_weights"])
    rows, acc_r, cm_r = run_full_comparison(model_r, cfg["name"])
    resample_results.extend(rows)
    resample_cms[cfg["name"]] = cm_r

    del model_r
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

resample_df = pd.DataFrame(resample_results)
resample_df.to_csv("/kaggle/working/resampling_comparison_perclass.csv", index=False)

overall_rows = resample_df[resample_df["Class"] == "Overall (weighted/macro)"]
overall_rows.to_csv("/kaggle/working/resampling_comparison_overall.csv", index=False)

print("\n=== Resampling Strategy Comparison — Overall ===")
print(overall_rows.to_string(index=False))
print("\n=== Resampling Strategy Comparison — Per-Class ===")
print(resample_df.to_string(index=False))

# ---- confusion matrices side-by-side ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, cm_) in zip(axes, resample_cms.items()):
    sns.heatmap(cm_, annot=True, fmt="d", cmap="Purples", xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("/kaggle/working/resampling_comparison_confusion.png", dpi=150)
plt.show()

# ---- grouped bar chart: F1 per class across all 3 strategies ----
plt.figure(figsize=(11, 5))
x = np.arange(len(CLASS_NAMES))
width = 0.25
colors = {"Class-Weighted (current/reference)": "#4C72B0", "Under-Sampling": "#C44E52", "Over-Sampling": "#55A868"}

for i, (name, _) in enumerate(resample_cms.items()):
    f1_vals = resample_df[(resample_df["Variant"] == name) & (resample_df["Class"].isin(CLASS_NAMES))]["F1 Score (%)"].values
    plt.bar(x + (i - 1) * width, f1_vals, width, label=name, color=colors[name])

plt.xticks(x, CLASS_NAMES)
plt.ylabel("F1 Score (%)")
plt.title("Per-Class F1: Class-Weighted vs Under-Sampling vs Over-Sampling")
plt.legend()
plt.tight_layout()
plt.savefig("/kaggle/working/resampling_comparison_f1_bar.png", dpi=150)
plt.show()